In [38]:
import os
import sys

# Spark가 사용할 파이썬 실행 파일 경로를 명확히 지정
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 윈도우에서 워커 접속 문제를 해결하기 위한 설정
os.environ['SPARK_LOCAL_IP'] = "127.0.0.1"

In [39]:
import pyspark
from pyspark.sql import SparkSession

In [40]:
# spark = SparkSession.builder \
#    .master("local[*]") \
#    .appName('test') \
#    .getOrCreate()

# SparkSession 설정 보강
#세션을 만들 때도 타임아웃 시간을 늘리고 바인딩 주소를 고정해줍니다.
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

In [41]:
#!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [42]:
!curl -L https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz -o fhvhv_tripdata_2021-01.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0

  3  123M    3 4381k    0     0  2209k      0  0:00:57  0:00:01  0:00:56 2209k
  9  123M    9 11.4M    0     0  3940k      0  0:00:32  0:00:02  0:00:30 7375k
 15  123M   15 18.6M    0     0  4796k      0  0:00:26  0:00:03  0:00:23 7357k
 20  123M   20 25.5M    0     0  5241k      0  0:00:24  0:00:04  0:00:20 7246k
 26  123M   26 32.4M    0     0  5550k      0  0:00:22  0:00:05  0:00:17 7207k
 30  123M   30 37.9M    0     0  5568k      0  0:00:22  0:00:06  0:00:16 6900k
 36  123M   36 45.2M    0     0  5797k      0  0:00:21  0:00:07  0:00:14 6905k
 41  123M   41 51.8M    0     0  5905k      0  0:0

In [43]:
#!gzip -dc fhvhv_tripdata_2021-01.csv.gz

In [44]:
#!wc -l fhvhv_tripdata_2021-01.csv

In [45]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [46]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [47]:
# head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [48]:
!powershell -command "Get-Content fhvhv_tripdata_2021-01.csv -TotalCount 1001 | Set-Content head.csv"

In [49]:
import pandas as pd

In [50]:
df_pandas = pd.read_csv('head.csv')

In [51]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [52]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

Integer - 4 bytes
Long - 8 bytes

In [53]:
from pyspark.sql import types

In [54]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [55]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [56]:
df = df.repartition(24)

In [57]:
#df.write.parquet('fhvhv/2021/01/') OR
df.write.parquet('fhvhv/2021/01/', mode='overwrite')

In [58]:
df = spark.read.parquet('fhvhv/2021/01/')

In [59]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [60]:
from pyspark.sql import functions as F

In [61]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-02 23:44:59|2021-01-03 00:13:08|         225|         178|   NULL|
|           HV0005|              B02510|2021-01-01 01:14:32|2021-01-01 01:26:38|         230|          79|   NULL|
|           HV0003|              B02869|2021-01-01 14:40:15|2021-01-01 14:52:31|          78|         174|   NULL|
|           HV0003|              B02869|2021-01-04 14:37:17|2021-01-04 14:52:20|         129|          95|   NULL|
|           HV0003|              B02878|2021-01-02 21:22:26|2021-01-02 21:26:26|         200|         200|   NULL|
|           HV0005|              B02510|2021-01-01 19:42:08|2021-01-01 19:54:25|

In [63]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [64]:
crazy_stuff('B02884')

's/b44'

In [65]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [67]:
# 파일명: 03_test_native.py
from pyspark.sql import functions as F

df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('num', F.expr("cast(substring(dispatching_base_num, 2, 10) as int)")) \
    .withColumn('base_id',
        F.when(F.col('num') % 7 == 0, F.concat(F.lit('s/'), F.lpad(F.hex(F.col('num')), 3, '0')))
        .when(F.col('num') % 3 == 0, F.concat(F.lit('a/'), F.lpad(F.hex(F.col('num')), 3, '0')))
        .otherwise(F.concat(F.lit('e/'), F.lpad(F.hex(F.col('num')), 3, '0')))
    ) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show(10)

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9CE| 2021-01-02|  2021-01-03|         225|         178|
|  e/9CE| 2021-01-01|  2021-01-01|         230|          79|
|  e/B35| 2021-01-01|  2021-01-01|          78|         174|
|  e/B35| 2021-01-04|  2021-01-04|         129|          95|
|  e/B3E| 2021-01-02|  2021-01-02|         200|         200|
|  e/9CE| 2021-01-01|  2021-01-01|         151|         116|
|  e/B3B| 2021-01-04|  2021-01-04|          42|         152|
|  e/9CE| 2021-01-01|  2021-01-01|          69|         174|
|  e/B3B| 2021-01-04|  2021-01-04|          91|         257|
|  a/B49| 2021-01-04|  2021-01-04|          74|          42|
+-------+-----------+------------+------------+------------+
only showing top 10 rows



In [69]:
#df \
#    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
#    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
#    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
#    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
#    .show()

In [72]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')


DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [73]:
#!head -n 10 head.csv
!powershell -command "Get-Content head.csv -TotalCount 10"

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
